# Earth-atmosphere modeling with NRLMSISE-00

**Universidad de Antioquia · Medios Continuos · 2025**

This notebook is the **atmospheric-modeling section (Exercise 3)** extracted from a larger university workshop. The Spanish explanations and original numerical workflow have been retained, except for these portfolio adaptations:

- The coursework's personal reference date was replaced with **June 15, 2025, at 12:00** to avoid sharing birth-date information. This is a new *example input*, not a claim that the old results were computed for that date.
- Cached outputs and specific narrative results from the original run were removed. **Re-run the cells in order** to regenerate results for the new date.
- The notebook is **not yet independently validated**; see [`REVIEW_NOTES.md`](../REVIEW_NOTES.md) for the known points to revisit.

**Scientific scope:** this models the **Earth's atmosphere**, not exoplanet habitability. It includes profiles, hydrostatic-balance comparison, a numerical tropopause estimate, global maps, and a simplified temperature-gradient fit.

# **Punto 3: Modelos Atmosféricos**

## a). Obtención de Datos Atmosféricos con el Modelo NRLMSISE-00

In [ ]:
!pip install -q nrlmsise00
!pip install -qU 'nrlmsise00[dataset]'
!pip install -q spaceweather cartopy

In [ ]:
import numpy as np
import spaceweather as sw
import cartopy.crs as ccrs
from datetime import datetime
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
from scipy.optimize import minimize
from matplotlib import animation, rc
from scipy.integrate import solve_ivp
from nrlmsise00.dataset import msise_4d
#NRLMSISE-00 usa datos del clima espacial.  Esos datos tienen que actualizarse
sw.update_data()

In [ ]:
# Definimos altura desde 0 hasta 100 km en pasos de 50 m
hmax = 100e3  # 100 km en metros
step = 50     # 50 metros
zs = np.arange(0, hmax + step, step)  # Array de alturas en metros

Se utilizó el modelo NRLMSISE-00 para obtener perfiles desde la superficie hasta 100 km de altitud en pasos de 50 m. La ubicación seleccionada fue El Cairo, Egipto (latitud 30.0444° N, longitud 31.2357° E), para el 15 de junio de 2025 a las 12:00.

El modelo proporcionó:

*   Temperatura ($T$)
*   Densidad total ($\rho$)
*   Composición química (He, N$_2$, O$_2$, O, Ar, etc.)


In [ ]:
# Fecha y hora: 15 junio 2025, 12:00
fecha = datetime(2025, 6, 15, 12, 0, 0)

# Ubicación: El Cairo, Egipto
lat = 30.0444   # grados norte
lon = 31.2357   # grados este

In [ ]:
# Obtenemos los datos del modelo NRLMSISE-00
ds = msise_4d(fecha, zs / 1e3, [lat], [lon])

In [ ]:
ds

## b). Cálculo de la presíon atmosférica y gráficos de perfiles

Podemos obtener la presión a cada altura con los valores de la temperatura$T$ y la densidad$ρ$, de modo que podemos caludlar la presión usando la ecuación de estado del gas ideal:

$$
P(z) = ρ(z) \frac{RT(z)}{M}
$$

donde $M$ es la masa molar promedio del aire, la cual es de aproximadamente $0.02897$ kg/mol.

In [ ]:
# Extraemos las variables del cubo de datos
temps = np.array(ds['Talt']).flatten() # Temperatura en K
rho = np.array(ds['rho']).flatten()  # Densidad en g/cm³
alts = np.array(ds['alt']).flatten() # Altura en km

# Convertimos la densidad a kg/m³
Rho = rho * 1000  # g/cm³ → kg/m³

R = 8.314462618 # J/(mol·K)
M = 28.97e-3    # kg/mol (aire)

# Calculamos presión usando la ecuación de estado P = rho * R * T / M
p = Rho * R * temps / M  # en Pa

In [ ]:
plt.figure(figsize=(14, 6))

# Temperatura vs Altura
plt.subplot(1, 3, 1)
plt.plot(temps, alts, color = 'crimson')
plt.xlabel('Temperatura (K)')
plt.ylabel('Altura (km)')
plt.title('Temperatura vs Altura')
plt.grid()

# Densidad vs Altura
plt.subplot(1, 3, 2)
plt.plot(Rho, alts, color = 'dodgerblue')
plt.xlabel('Densidad (kg/m³)')
plt.title('Densidad vs Altura')
plt.grid()

# Presión vs Altura (en hPa)
plt.subplot(1, 3, 3)
plt.plot(p / 100, alts, color = 'seagreen')
plt.xlabel('Presión (hPa)')
plt.title('Presión vs Altura')
plt.grid()

plt.tight_layout()
plt.show()


Podemos observar que las gráficas muestran una disminución de la temperatura en la troposfera.

Vemos además, que el perfil de temperatura muestra cómo varía la estructura de la atmósfera en capas, comenzando por la troposfera, donde la temperatura disminuye con la altura hasta alcanzar un mínimo en la tropopausa, para luego aumentar en la estratosfera debido a la absorción de radiación por la capa de ozono hasta un máximo en la estratopausa, después la temperatura vuelve a descender en la mesosfera hasta su punto más frío en la mesopausa y finalmente se eleva de nuevo en la termosfera.

Por otro lado, las gráficas de densidad y presión muestran un comportamiento máximo en la superficie y disminuyen de forma exponencial con la altitud,pues la gravedad concentra la mayor parte de las moléculas de aire en las capas más bajas, generando así mayor densidad y mayor presión cerca del suelo.

## c). Evaluación del equilibrio hidrostático

Para determianrr si la atmósfera está en equilibrio hidrostático debeos comprobar que se cumpla la ecuación

$$
\frac{dP}{dz} = - ρ(Z)G(Z)
$$

donde $g(z)$ es la aceleración de la gravedad en función de la altura:

$$
g(z) = \frac{GM}{(R+z)^2}
$$

In [ ]:
# Constantes
G = 6.67430e-11 # m³/kg/s²
M = 5.972e24 # kg
R = 6.371e6  # m

# Altura en metros
z_m = alts * 1000

# Calculamos g(z)
g_z = G * M / (R + z_m)**2

# Derivada de la presión respecto a z
dp_dz_num = np.gradient(p, z_m)

# Ecuación hidrostática: -ρ * g
dp_dz_hydro = -Rho * g_z

# Realizamos la comparación
plt.figure(figsize=(8, 6))
plt.plot(dp_dz_num, z_m / 1000, label='Numérico', color = 'crimson')
plt.plot(dp_dz_hydro, z_m / 1000, '--', label='Teórico', color = 'dodgerblue')
plt.xlabel('Gradiente de presión (Pa/m)')
plt.ylabel('Altura (km)')
plt.title('Verificación del equilibrio hidrostático')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


La atmósfera calculada cumple con el equilibrio hidrostático, como se observa en la comparación entre el gradiente de presión obtenido numéricamente y el valor teórico derivado de la ecuación
$\frac{dp}{dz} = -\rho(z)\,g(z)$, pues en cada nivel de altitud, la presión disminuye  para compensar el peso de la columna de aire situada por encima, manteniendo así el equilibrio vertical.

##d). Altura de la tropopausa

La tropopausa se define como la altura a la cual la temperatura deja de disminuir con la altura, es decir, el primer punto donde:

$$
\frac{dT}{dz}\geq 0
$$

Por lo tanto, al examinar el perfil de temperatura obtenido a partir del modelo atmosférico, se encuentra que la altura de la tropopausa corresponde al punto en el que la pendiente de la curva de temperatura con respecto a la altitud deja de ser negativa y se vuelve positiva, esta altitud marca el límite superior de la troposfera y el inicio de la estratósfera.

In [ ]:
# Calculamos dT/dz numéricamente
dT_dz = np.gradient(temps, z_m)  # K/m

# Encontramos el índice donde dT/dz pasa de negativo a cero o positivo
tropo_idx = np.argmax(dT_dz >= 0)  # primer lugar donde ya no decrece

# Obtenemos la altura en km
tropopause_alt_km = z_m[tropo_idx] / 1000

In [ ]:
print(f"Altura estimada de la tropopausa: {tropopause_alt_km:.2f} km")
plt.figure(figsize=(8, 6))
plt.plot(temps, z_m / 1000, label='Temperatura', color = 'dodgerblue')
plt.axhline(tropopause_alt_km, color='crimson', linestyle='--', label=f'Tropopausa = {tropopause_alt_km:.2f} km')
plt.xlabel('Temperatura (K)')
plt.ylabel('Altura (km)')
plt.title('Perfil de Temperatura y Tropopausa')
plt.legend()
plt.grid()
plt.show()

La altura estimada de la tropopausa se obtiene del perfil de temperatura calculado en la ejecución actual. El valor numérico debe interpretarse para los parámetros de entrada de esta versión.

## e). Mapa

### Gráfico de contornos

Para representar la variación espacial de la altura de la tropopausa, se construyó un gráfico de contornos sobre un mapa global, utilizando como variables la latitud y la longitud. Para cada punto de la grilla (definida en términos de latitud y longitud), se calculó el perfil vertical de temperatura y se identificó la altitud en la que la temperatura deja de disminuir con la altura, es decir, la altura de la tropopausa. Estos valores se organizaron en una matriz que fue graficada usando un mapa con proyección Robinson, mostrando cómo la altura de la tropopausa varía alrededor del planeta.

In [ ]:
# Parámetros de altura
hmax = 25e3   # limitamos a 25 km para la tropopausa
step = 100     # pasos de 100 m para perfilar
zs = np.arange(0, hmax + step, step) / 1e3  # en km

In [ ]:
# Definimos la rejilla global (latitud y longitud)
lats = np.linspace(-90, 90, 10)
lons = np.linspace(-180, 180, 20)

# Fecha y hora fijos
fecha = datetime(2025, 6, 15, 12, 0, 0)

# Array para guardar alturas de tropopausa
tropo_heights = np.zeros((len(lats), len(lons)))

# Bucle sobre la rejilla
for i, lat in enumerate(lats):
  for j, lon in enumerate(lons):
    ds = msise_4d(fecha, zs, [lat], [lon])
    T = np.array(ds['Talt']).flatten()
    altitudes = np.array(ds['alt']).flatten()
    # Gradiente de temperatura
    dT_dz = np.gradient(T, altitudes)
    # Tropopausa: donde el gradiente se hace >= 0 (o muy pequeño)
    idx = np.where(dT_dz >= -0.001)[0]
    if len(idx) == 0:
      tropo_heights[i, j] = np.nan
    else:
      tropo_heights[i, j] = altitudes[idx[0]]

In [ ]:
# Mapa
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.gridlines(draw_labels=True)

LONS, LATS = np.meshgrid(lons, lats)
contour = ax.contourf(LONS, LATS, tropo_heights,
                      transform=ccrs.PlateCarree(),
                      levels=20, cmap='viridis')

cbar = plt.colorbar(contour, orientation='horizontal', pad=0.05)
cbar.set_label('Altura de la tropopausa [km]')
plt.title('Mapa global de altura de la tropopausa estimada')
plt.show()

### Animación

Adicionalmente, se generó una animación que muestra la evolución de la temperatura en la superficie de la Tierra a lo largo de un día completo. Para esto, se calcularon las temperaturas superficiales para cada hora del día y en cada punto de la grilla global. Estas temperaturas se representaron mediante mapas de contornos para cada instante de tiempo y luego se animaron secuencialmente, visualizando así cómo cambia la distribución térmica superficial en diferentes regiones del planeta a medida que transcurren las horas del día.

In [ ]:
# Usar HTML-JS para mostrar animaciones en el notebook
rc('animation', html='jshtml')

# Rejilla global
lats = np.linspace(-90, 90, 10)
lons = np.linspace(-180, 180, 10)
horas = np.arange(24)

def calcular_temperatura_vec(lats, lons, horas):
  zs = np.array([0.0])  # altitud fija (superficie)
  nl, nm, nh = len(lats), len(lons), len(horas)
  T_surf = np.empty((nl, nm, nh))
  for h, hora in enumerate(horas):
    ds = msise_4d(datetime(2025, 6, 15, int(hora), 0, 0), zs, lats, lons)
    T_surf[:, :, h] = ds["Talt"][0, :, :]
  return T_surf

Temperaturas = calcular_temperatura_vec(lats, lons, horas)
LONS, LATS = np.meshgrid(lons, lats)
levels = np.linspace(Temperaturas.min(), Temperaturas.max(), 20)

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax  = plt.axes(projection=ccrs.Robinson())

def update(frame):
  ax.clear()
  ax.set_global()
  ax.coastlines(resolution='110m')
  ax.gridlines()
  cont = ax.contourf(
      LONS, LATS, Temperaturas[:, :, frame],
      levels=levels, cmap='viridis',
      transform=ccrs.PlateCarree(), extend='both'
  )
  ax.set_title(f'Temperatura superficial {frame:02d}:00 UTC')

anim = animation.FuncAnimation(
  fig, update,
  frames=24,
  interval=500,
  blit=False
)

anim

## f). Modelo homentrópico

Se ajusta un modelo homentrópico a los perfiles de la troposfera. Este modelo simplificado depende de tres parámetros: la temperatura superficial ($T_0$), el índice politrópico ($\gamma$) y la altura de escala ($h_0$) que define el tope del modelo. El ajuste se realiza minimizando la función $χ^2$

Las ecuaciones del modelo homentrópico son:

\begin{align*}
    T_{\mathrm{hom}}(z)   &= T_0 \left(1 - \frac{z}{h_0}\right) \\
    \rho_{\mathrm{hom}}(z) &= \rho_0 \left(1 - \frac{z}{h_0}\right)^{\frac{1}{\gamma - 1}} \\
    p_{\mathrm{hom}}(z)   &= p_0 \left(1 - \frac{z}{h_0}\right)^{\frac{\gamma}{\gamma - 1}}
\end{align*}
donde:

* $T_0$ es la temperatura superficial (en $z = 0$),
* $\rho_0$ es la densidad superficial,
* $p_0$ es la presión superficial,
* $h_0$ es la escala característica de altura,
* $\gamma$ es el índice adiabático.

Se consideraron los perfiles extraidos anteriormente para El Cairo el 15 de junio de 2025 de presión, densidad y temperatura. Estos perfiles se denotan como $T(z_i)$, $\rho(z_i)$ y $p(z_i)$.


In [ ]:
R = 8.314462618     # J/(mol·K)
M = 28.97e-3        # kg/mol (aire seco)
R_especifica = R/M  # J/(kg·K)
g = 9.81            # m/s²

# Datos extraídos con el modelo NRLMSISE00 para El Cairo el día 15 de junio de 2025
alts_km = np.array(ds['alt']).flatten()  # Altura en km
z_i = alts_km * 1000                     # Convertimos a metros
temps = np.array(ds['Talt']).flatten()   # Temperatura en K
rho = np.array(ds['rho']).flatten()      # Densidad en g/cm³
Rho_i = rho * 1000                       # Convertimos a kg/m³
p_i = Rho_i * R_especifica * temps       # en Pa

In [ ]:
# Calculamos los perfiles homentrópicos de T, p y rho
def modelo_homentropico(z, T0, gamma):
  T = T0 - gamma * z
  #Valores de la superficioe
  idx_surface = np.argmin(np.abs(z))
  p0 = p_i[idx_surface]
  rho0 = Rho_i[idx_surface]
  exponente = g / (R_especifica*gamma)
  p = p0 * (T/T0)**exponente
  rho = rho0 * (T/T0)**(exponente-1)
  T[T<0] = np.nan
  p[np.isnan(T)] = np.nan
  rho[np.isnan(T)] = np.nan

  return T, p, rho

In [ ]:
T_h, p_h, rho_h = modelo_homentropico(z_i, 300, 0.0065)

plt.figure(figsize=(15, 5))

#Perfil de temperatura usando el modelo homentrópico
plt.subplot(1, 3, 1)
plt.plot(temps, alts_km, color = 'crimson', label = 'Modelo teórico')
plt.plot(T_h, alts_km, 'g--', label='Modelo homentrópico')
plt.xlabel('Temperatura (K)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Temperatura')
plt.legend()

# Perfil de presión usando el modelo hometrópico
plt.subplot(1, 3, 2)
plt.plot(p_i/100, alts_km, color = 'dodgerblue', label = 'Modelo teórico')
plt.plot(p_h/100, alts_km, 'b--', label='Modelo homentrópico')
plt.xlabel('Presión (hPa)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Presión')
plt.legend()


# Perfil de densidad usando el modelo homentrópico
plt.subplot(1, 3, 3)
plt.plot(Rho_i, alts_km, color = 'darkorange', label = 'Modelo teórico')
plt.plot(rho_h, alts_km, 'r--', label='Modelo homentrópico')
plt.xlabel('Densidad (kg/m³)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Densidad')
plt.legend()

**Método de ajuste**

Se minimizó la siguiente función:
$$
\chi^2(h_0, \gamma, T_0) =
\sum_{i} \left( \frac{T_{\mathrm{hom}}(z_i) - T(z_i)}{T(z_i)} \right)^2
+ \sum_{i} \left( \frac{\rho_{\mathrm{hom}}(z_i) - \rho(z_i)}{\rho(z_i)} \right)^2
+ \sum_{i} \left( \frac{p_{\mathrm{hom}}(z_i) - p(z_i)}{p(z_i)} \right)^2
$$

Los parámetros libres del modelo son $h_0$, $\gamma$ y $T_0$, con valores inciales y límites similares a los vistos en clase.

La minimización se realizó mediante el método L-BFGS-B de scipy.optimize.minimize.

In [ ]:
def chi2(params, z, T, p, rho):
  T0, gamma = params
  T_hom, p_hom, rho_hom = modelo_homentropico(z, T0, gamma)
  # Valores válidos
  valid = ~np.isnan(T_hom)
  # Calculamos chi2 para cada variable
  chi2_T = np.sum(((T_hom[valid] - T[valid]) / T[valid])**2)
  chi2_p = np.sum(((p_hom[valid] - p[valid]) / p[valid])**2)
  chi2_rho = np.sum(((rho_hom[valid] - rho[valid]) / rho[valid])**2)

  return chi2_T + chi2_p + chi2_rho

In [ ]:
# Valores iniciales
T0 = temps[np.argmin(np.abs(z_i))]  # Temperatura a la altitud más baja
gamma0 = 0.0065

# Minimizamos teniendo en cuenta las restricciones
bounds = [(max(250, T0-20), min(350, T0+20)), # Rango para T0
          (0.005, 0.01)] # Rango para el gradiente en K/m

# Minimizamos usando los datos de la troposfera ( troposfera aproximada)
limite_tropo_idx = np.where(alts_km > 16)[0][0]
z_fit = z_i[:limite_tropo_idx]
T_fit = temps[:limite_tropo_idx]
p_fit = p_i[:limite_tropo_idx]
rho_fit = Rho_i[:limite_tropo_idx]

# Minimizamos
result = minimize(chi2, [T0, gamma0],
                  args = (z_fit, T_fit, p_fit, rho_fit),
                  method = 'L-BFGS-B',
                  bounds=bounds)

T0_opt, gamma_opt = result.x
h0_opt = T0_opt/gamma_opt

In [ ]:
print(f"Temperatura superficial (T0): {T0_opt:.2f} K  ({T0_opt - 273.15:.2f} °C)")
print(f"Gradiente térmico (L): {gamma_opt * 1000:.4f} K/km")
print(f"Altura de escala (h0): {h0_opt / 1000:.2f} km")
print(f"Valor mínimo de χ²: {result.fun:.2f}")

In [ ]:
# Calculamos los perfiles del modelo homentrópico
T_opt, p_opt, rho_opt = modelo_homentropico(z_i, T0_opt, gamma_opt)

plt.figure(figsize=(15, 5))
# Perfil de Temperatura
plt.subplot(1, 3, 1)
plt.plot(temps, alts_km, color = 'crimson', label='Modelo empírico')
plt.plot(T_opt, alts_km, 'g--', label='Modelo homentrópico mínimo χ²')
plt.ylim(0,20)
plt.xlabel('Temperatura (K)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Temperatura')
plt.grid(True)
plt.legend()

# Perfil de Presión
plt.subplot(1, 3, 2)
plt.plot(p_i/100, alts_km, color = 'dodgerblue', label='Modelo empírico')  # Convertimos a hPa
plt.plot(p_opt/100, alts_km, 'b--', label='Modelo homentrópico mínimo χ²')
plt.ylim(0,20)
plt.xlabel('Presión (Pa)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Presión')
plt.grid(True)
plt.legend()

# Perfil de Densidad
plt.subplot(1, 3, 3)
plt.plot(Rho_i, alts_km, color = 'darkorange', label='Modelo empírico')
plt.plot(rho_opt, alts_km, 'r--', label='Modelo homentrópico mínimo χ²')
plt.ylim(0,20)
plt.xlabel('Densidad (kg/m³)')
plt.ylabel('Altitud (km)')
plt.title('Perfil de Densidad')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

**Resultados del ajuste.** Tras ejecutar de nuevo el notebook, utiliza los valores impresos por el código para describir el mejor ajuste. Los resultados numéricos de la ejecución original no se reproducen aquí porque se cambió la fecha de entrada para proteger la privacidad.

##g). Múltiples ubicaciones sobre la superficie terrestre

In [ ]:
# Créditos:
#   Algunas secciones de este script (en particular las funciones
#   del ciclo for) fueron generadas con la
#   asistencia de ChatGPT (modelo OpenAI o3) el 14-jul-2025.

import warnings

# Ignorar advertencias de runtime que no son críticas
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 1. Configuración de la Rejilla Global ---
lats = np.linspace(-90, 90, 21)  # Aumentamos un poco la resolución
lons = np.linspace(-180, 180, 41)
fecha = datetime(2025, 6, 15, 12, 0, 0)
zs_km = np.arange(0, 15.1, 0.5)

# Arrays para guardar los resultados
T0_map = np.full((len(lats), len(lons)), np.nan)
gamma_map = np.full((len(lats), len(lons)), np.nan)
h0_map = np.full((len(lats), len(lons)), np.nan)

# --- 2. Bucle Principal (Versión Final) ---
print("Iniciando el cálculo final de mapas globales...")
for i, lat in enumerate(lats):
    print(f"Procesando fila {i+1}/{len(lats)} (Latitud: {lat:.1f}°)...")
    for j, lon in enumerate(lons):
        try:
            # Obtener perfil atmosférico
            ds = msise_4d(fecha, zs_km, [lat], [lon])
            T_data = ds['Talt'].values.flatten()
            z_data_m = ds['alt'].values.flatten() * 1000

            if np.isnan(T_data).any():
                continue

            # Calcular T0 y Gradiente L
            T0_opt = T_data[0]
            limite_idx = np.searchsorted(z_data_m, 10000)
            if limite_idx < 4:
                continue

            pendiente, _ = np.polyfit(z_data_m[:limite_idx], T_data[:limite_idx], 1)
            gamma_opt = -pendiente

            # Filtro de seguridad para evitar valores no físicos en los polos
            if 0.001 < gamma_opt < 0.012:
                T0_map[i, j] = T0_opt
                gamma_map[i, j] = gamma_opt
                h0_map[i, j] = T0_opt / gamma_opt

        except Exception:
            # Si un punto específico falla por cualquier razón, lo saltamos
            continue

def create_global_map(data, lons, lats, title, cbar_label, cmap='viridis'):
  fig = plt.figure(figsize=(12, 6))
  ax = plt.axes(projection=ccrs.Robinson())
  ax.set_global()
  ax.coastlines()
  ax.gridlines(alpha=0.5, linestyle='--')

  LONS, LATS = np.meshgrid(lons, lats)
  # Usamos pcolormesh en lugar de contourf para manejar mejor los NaNs
  # y evitar huecos en el mapa
  mesh = ax.pcolormesh(LONS, LATS, data, transform=ccrs.PlateCarree(), cmap=cmap, shading='auto')

  cbar = plt.colorbar(mesh, orientation='horizontal', pad=0.08, shrink=0.8)
  cbar.set_label(cbar_label)
  plt.title(title)
  plt.show()

In [ ]:
create_global_map(T0_map, lons, lats, 'Mapa Global de Temperatura Superficial ($T_0$)', '$T_0$ [K]', cmap='plasma')

**Interpretación pendiente de la nueva ejecución.** Inspecciona el mapa resultante y relaciona los contrastes latitudinales con la fecha y las condiciones del modelo. Las observaciones escritas para la fecha original no se trasladan automáticamente a esta versión.

In [ ]:
create_global_map(gamma_map * 1000, lons, lats, 'Mapa Global de Gradiente Térmico ($L$)', '$L$ [K/km]', cmap='inferno')

El gradiente térmico (la rapidez con que la temperatura desciende con la altura) muestra la dependencia latitudinal, con valores más altos cerca del ecuador (7 K/km) y más bajos cerca de los polos (5 K/km), lo cual está relacionado con la variación en la altura de la tropopausa, que es más alta cerca del ecuador, lo que requiere un mayor gradiente para una misma diferencia de temperatura. Esto nos muestra cómo la radiación solar y la actividad convectiva influyen en los gradientes térmicos, con un mayor gradiente en áreas subtropicales, un gradiente alto significa que la atmósfera es más inestable y convectiva. Esto ocurre en las regiones cálidas donde la superficie calienta intensamente el aire, provocando que suba y se enfríe rápidamente.



In [ ]:
create_global_map(h0_map / 1000, lons, lats, 'Mapa Global de Altura de Escala ($h_0$)', '$h_0$ [km]', cmap='viridis')

Sabemos que el parámetro $h_0$ representa la altura o "espesor" característico de la troposfera. Vemos que el mapa confirma un hecho fundamental de la atmósfera terrestre: la troposfera es significativamente más alta en los trópicos y más achatada en los polos.